# 4-1. NPLM — 확률적 언어 모델의 구조

> **이 노트북에서 답할 질문**
> Bengio(2003)의 뉴럴 확률 언어 모델은 정확히 어떤 구조이고, 왜 그 구조인가?

## 1. 역사적 위치

2003년 Bengio 팀의 *A Neural Probabilistic Language Model* 은
지금 우리가 쓰는 모든 LLM의 조상입니다. 논문이 제시한 것은 두 가지였습니다.

1. **단어를 학습 가능한 벡터로 표현하자** (= 임베딩)
2. **그 벡터들로 다음 단어의 확률을 계산하는 신경망을 쓰자**

3-3에서 만든 `TinyLM`이 사실 이 구조의 축소판이었습니다.
이번엔 논문 구조를 제대로, 그리고 **평가 손실까지 재면서** 만들어 봅니다.

## 2. 구조

$$\hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

```
입력: 앞 n-1개 단어의 인덱스          (B, n-1)
  │
  ├─ 임베딩 조회 E                     (B, n-1, d)
  │
  ├─ 이어붙이기 (concatenate)          (B, (n-1)*d)
  │        x
  ├──────────────────────────┐
  │                          │
  ├─ 은닉층  tanh(Hx + b)     │ 직접 연결 Wx
  │        (B, h)            │ (논문의 선택 요소)
  │                          │
  └─ 출력층  U·tanh(...) + Wx + b      (B, V)
             │
           softmax
             ↓
       다음 단어의 확률 분포
```

수식으로 쓰면:

$$y = b + Wx + U \tanh(d_{\text{bias}} + Hx)$$

- $x$: 임베딩을 이어붙인 벡터
- $H, U$: 은닉층 가중치
- $W$: **직접 연결**. 임베딩에서 출력으로 바로 가는 지름길
- 최종적으로 $\hat{P} = \text{softmax}(y)$

### 직접 연결($Wx$)이 왜 있나?

깊은 경로($\tanh$ 를 거치는 길)는 복잡한 패턴을,
지름길은 단순한 패턴을 담당합니다. 학습 초반에 지름길이 먼저 학습되어 수렴을 돕습니다.

> 이 아이디어가 나중에 **잔차 연결(residual connection)** 로 이어집니다.
> 트랜스포머의 `x + Attention(x)` 가 바로 같은 발상입니다. (Ch5에서 다시 만납니다.)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import load_sentences, set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

## 3. 데이터 준비 — 이번엔 학습/평가를 나눕니다

3-2에서 확인했듯, 학습 손실만 보면 과적합을 놓칩니다.
처음부터 나눠 놓고 시작합니다.

In [ ]:
import random

def tokenize(s):
    return s.replace(".", " .").split()

BOS = "<s>"
sents = load_sentences()

# 어휘는 학습 문장에서만 만들어야 공정합니다 (평가 문장을 미리 보면 반칙)
shuffled = sents[:]
random.Random(0).shuffle(shuffled)
cut = int(len(shuffled) * 0.8)
train_sents, val_sents = shuffled[:cut], shuffled[cut:]

UNK = "<unk>"
vocab = [BOS, UNK] + sorted({t for s in train_sents for t in tokenize(s)})
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)

print(f"학습 {len(train_sents)}문장 / 평가 {len(val_sents)}문장")
print(f"어휘 V = {V}  (평가에만 나오는 단어는 <unk> 처리)")

In [ ]:
N = 4            # n-gram 크기. 앞 N-1 = 3개를 보고 다음 1개를 예측
CTX = N - 1

def make_dataset(sent_list):
    xs, ys = [], []
    for s in sent_list:
        toks = [BOS] * CTX + tokenize(s)
        ids = [stoi.get(t, stoi[UNK]) for t in toks]
        for i in range(CTX, len(ids)):
            xs.append(ids[i - CTX:i])
            ys.append(ids[i])
    return torch.tensor(xs), torch.tensor(ys)

Xtr, Ytr = make_dataset(train_sents)
Xva, Yva = make_dataset(val_sents)
print(f"학습 샘플 {len(Xtr)}개, 평가 샘플 {len(Xva)}개")
print(f"예시: {[itos[i.item()] for i in Xtr[5]]}  ->  {itos[Ytr[5].item()]}")

Xtr, Ytr = Xtr.to(device), Ytr.to(device)
Xva, Yva = Xva.to(device), Yva.to(device)

## 4. 모델 구현

논문 구조를 그대로 옮깁니다. 직접 연결은 켜고 끌 수 있게 만들어서
**정말 도움이 되는지 직접 확인**해 보겠습니다.

In [ ]:
class NPLM(nn.Module):
    def __init__(self, V, d=64, hidden=128, ctx=CTX, direct=True):
        super().__init__()
        self.emb = nn.Embedding(V, d)                  # E : V x d
        self.H = nn.Linear(ctx * d, hidden)            # H : 은닉층
        self.U = nn.Linear(hidden, V)                  # U : 은닉 -> 출력
        self.W = nn.Linear(ctx * d, V, bias=False) if direct else None   # 직접 연결
        self.direct = direct

    def forward(self, idx):                 # idx : (B, ctx)
        x = self.emb(idx).flatten(1)        # (B, ctx*d)  <- 이어붙이기
        y = self.U(torch.tanh(self.H(x)))   # 깊은 경로
        if self.direct:
            y = y + self.W(x)               # 지름길을 더함
        return y                            # (B, V) logits

m = NPLM(V)
print(m)
print(f"\n파라미터 수: {human(count_params(m))}")

### 파라미터가 어디에 몰려 있나

LLM을 다룰 때 반드시 붙는 감각입니다. **파라미터는 거의 항상 임베딩과 출력층에 몰립니다.**

In [ ]:
for name, p in m.named_parameters():
    if p.dim() > 1:
        print(f"{name:<10} {str(tuple(p.shape)):>14}  {human(p.numel()):>9}  ({p.numel()/count_params(m):5.1%})")
print()
print("V(어휘)가 곱해지는 층들이 대부분을 차지합니다.")
print("실제 LLM에서 어휘 크기를 함부로 못 늘리는 이유이기도 합니다.")

## 5. 학습 — 평가 손실을 함께 봅니다

In [ ]:
@torch.no_grad()
def evaluate(model, X, Y):
    model.eval()
    loss = F.cross_entropy(model(X), Y).item()
    model.train()
    return loss, math.exp(loss)

def train(model, steps=1500, lr=3e-3, log_every=250):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(1, steps + 1):
        loss = F.cross_entropy(model(Xtr), Ytr)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % log_every == 0 or step == 1:
            vl, vppl = evaluate(model, Xva, Yva)
            history.append((step, loss.item(), vl))
            print(f"step {step:5d} | train {loss.item():6.3f} (ppl {math.exp(loss.item()):7.2f})"
                  f" | valid {vl:6.3f} (ppl {vppl:7.2f})")
    return history

set_seed(1234)
model = NPLM(V, d=64, hidden=128, direct=True)
print(f"기준선: 아무것도 모르는 모델의 손실 = log(V) = {math.log(V):.3f}, 혼란도 = {V}\n")
hist = train(model)

### 로그를 읽는 법 — 여기가 이 노트북의 핵심입니다

방금 본 숫자는 **교과서적인 과적합**입니다. 찬찬히 보세요.

- **train 손실**: $\log V \approx 5.5$ 에서 0.42까지 곤두박질 → 혼란도 1.5, 거의 완벽
- **valid 손실**: 6.2에서 시작해서 **올라가기만** 합니다 → 혼란도 수천

모델은 학습 문장 400여 개를 **통째로 외웠고**, 그 외운 지식이 새 문장에는 전혀 쓸모가 없습니다.
3-2에서 예고한 바로 그 현상입니다.

**왜 이렇게까지 심한가?** 파라미터가 116K인데 학습 샘플이 400개뿐입니다.
샘플 하나당 파라미터 290개. 외우지 않는 게 이상하죠.

> **이건 모델이 잘못 짜인 게 아니라 데이터가 없어서입니다.**
> 언어 모델링에서 데이터 규모는 선택 사항이 아니라 전제 조건입니다.
> 이 노트북에서 배울 것은 "좋은 성능"이 아니라 **구조와, 과적합을 알아보는 눈**입니다.

실무에서는 이럴 때 이렇게 합니다:
1. 데이터를 늘린다 (가장 확실)
2. 모델을 줄인다 (d, hidden 축소)
3. 드롭아웃/가중치 감쇠를 넣는다
4. valid 손실이 최저일 때 멈춘다 (early stopping)

## 6. 직접 연결은 정말 도움이 되나

In [ ]:
def loss_after(direct, steps):
    set_seed(1234)
    mdl = NPLM(V, d=64, hidden=128, direct=direct).to(device)
    opt = torch.optim.Adam(mdl.parameters(), lr=3e-3)
    for _ in range(steps):
        l = F.cross_entropy(mdl(Xtr), Ytr)
        opt.zero_grad(); l.backward(); opt.step()
    return l.item(), count_params(mdl)

print(f"{'스텝':>6} | {'직접 연결 없음':>14} | {'직접 연결 있음':>14}")
print("-" * 42)
for steps in (20, 40, 60, 100, 300):
    a, _ = loss_after(False, steps)
    b, _ = loss_after(True, steps)
    print(f"{steps:>6} | {a:>14.4f} | {b:>14.4f}")

n_no  = count_params(NPLM(V, d=64, hidden=128, direct=False))
n_yes = count_params(NPLM(V, d=64, hidden=128, direct=True))
print(f"\n파라미터: 없음 {human(n_no)}  ->  있음 {human(n_yes)}  ({n_yes/n_no:.1f}배)")

### 읽는 법

**스텝이 적을 때 차이가 극적입니다.** 20스텝에서 직접 연결이 있으면 손실이 1/3 수준입니다.
하지만 충분히 학습하면 둘 다 같은 곳에 도착합니다.

즉 직접 연결의 효과는 **최종 성능이 아니라 수렴 속도**입니다.
대신 파라미터를 크게 늘립니다($V \times \text{ctx} \cdot d$ 짜리 행렬이 통째로 추가).

> 현대 구조는 이 아이디어를 **잔차 연결**로 바꿉니다.
> $y = f(x) + x$ — 가중치 없이 그냥 더하기만 해서 같은 효과를 냅니다. **파라미터 0개.**
> Ch5에서 트랜스포머 블록을 만들 때 이 줄을 직접 쓰게 됩니다.

## 7. NPLM의 한계 — 왜 트랜스포머가 필요한가

NPLM은 N-gram의 희소성을 풀었지만, **구조적 한계 두 가지**가 남았습니다.

### 한계 1: 문맥 길이가 고정

`ctx*d` 크기의 입력을 받는 `nn.Linear` 가 박혀 있습니다.
문맥을 3개에서 100개로 늘리려면 **모델 구조 자체를 바꾸고 처음부터 다시 학습**해야 합니다.

In [ ]:
d, hidden = 64, 128
print(f"{'문맥 길이':>10} | {'입력 차원':>10} | {'H층 파라미터':>14} | {'W(직접) 파라미터':>18}")
print("-" * 62)
for ctx in (3, 8, 32, 128, 1024):
    inp = ctx * d
    print(f"{ctx:>10} | {inp:>10,} | {human(inp*hidden):>14} | {human(inp*V):>18}")
print()
print("문맥 길이에 파라미터가 '선형으로' 붙습니다. 1024 토큰을 보려면 감당이 안 됩니다.")

### 한계 2: 위치가 고정되어 유연하지 않다

이어붙이기(concatenate)는 **"몇 번째 자리에 있는 단어인가"로 가중치가 갈립니다.**

`나는 어제 밥을 먹었다` 에서 `밥을`은 3번째 자리 가중치를 씁니다.
`어제 나는 밥을 먹었다` 에서도 3번째지만,
`나는 밥을 먹었다` 에서는 2번째 자리 가중치를 쓰게 됩니다.

같은 단어의 같은 역할인데 **다른 파라미터가 처리**합니다. 대단히 비효율적이죠.

### 그래서 필요한 것

> **문맥 길이에 상관없이 작동하고, 중요한 단어를 위치와 무관하게 골라내는 장치**

그것이 **어텐션**이고, 그것으로 만든 것이 **트랜스포머**입니다.

---

## 정리

| | N-gram | NPLM | 트랜스포머 |
|---|---|---|---|
| 단어 표현 | 원-핫(직교) | 임베딩 | 임베딩 |
| 처음 보는 문맥 | 불가 | 가능 | 가능 |
| 문맥 길이 | 고정(짧음) | 고정 | **가변(길게)** |
| 위치 처리 | 자리별 카운트 | 자리별 가중치 | 위치 인코딩 + 어텐션 |

## 연습

1. `N` 을 3, 5, 7 로 바꿔 가며 valid 혼란도를 비교하세요. 개선되다 멈추는 지점이 있나요?
2. `hidden` 을 32/128/512 로 바꿔 보세요. 파라미터와 성능의 교환비가 보이나요?
3. 학습 중 valid 손실이 최저인 시점의 모델을 저장하는 코드를 추가하세요 (early stopping).

## 다음

→ `0402_Tokenization.ipynb` : "단어"를 어떻게 자를 것인가 — 한국어의 진짜 난관